# Exercise 2. LoRa for Low-Resource Languages
NLP for social good is not just about reducing harmful outputs; it is also about making AI accessible across languages, not only English. Low- and medium-resource languages, from Nigerian Pidgin to Danish, are often left behind. 

```{figure} ../figures/class8/neural-space-low-resource.png
---
name: neural-space-low-resource
width: 100%
---
Fig. borrowed from [NeuralSpace blogpost](https://medium.com/neuralspace/challenges-in-using-nlp-for-low-resource-languages-and-how-neuralspace-solves-them-54a01356a71b) by Felix Laumann
```

Fine-tuning LLMs can help, but it is costly. LoRA (Low-Rank Adaptation) offers a parameter-efficient alternative, reducing trainable parameters by up to 10,000 times. In other words, rather than training all 8 billion parameters of a model like [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B-Base), LoRA updates only a small fraction.

## 2.1 Intro to LoRa?
If you're interested in the math behind LoRa (but in an intuitive way), I encourage you to read Sebastian Raschka's [blogpost](https://magazine.sebastianraschka.com/i/138081202/a-brief-introduction-to-lora). You can also read the original paper by {cite:t}`hu_lora_2021`. 

In the code implementation ([PEFT]() library), LoRa is treated as a sort of "adapter" that you can train and keep seperately, essentially allowing you to place it on other models (if the base architecture matches):

```{figure} ../figures/class8/lora_adapter.png
---
name: lora_adapter
width: 100%
---
From HF's [smol course](https://huggingface.co/learn/smol-course/en/unit1/3a)
```
:::{admonition} What are Adapters? Is LoRa Really an Adapter?
:class: dropdown, tip
Adapters are extra trainable parameters that you add to a model, keeping its own model weights frozen. While the original paper does not call LoRa an adapter {cite:p}`hu_lora_2021`, the term is used everywhere. 

Read Jason Phang's take on this: [Should we consider LoRa an Adapter?](https://jasonphang.com/posts/2023/07/post1/).   
-> TL:DR; Historically, LoRa is not an adapter, but it probably can be considered one.
:::

## 2.2 Setup
For the code implementation, we'll use the [PEFT](https://huggingface.co/docs/peft/en/index) and [TRL](https://huggingface.co/docs/trl/en/index) library by Hugging Face
```bash
source .venv/bin/activate
pip install peft trl
```

If you don't already have this in your `venv`, we also need:
```bash
pip install transformers datasets torch
```

Let's import:

In [324]:
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import load_dataset
from peft import LoraConfig, AutoPeftModelForCausalLM
import torch

from trl import SFTTrainer, SFTConfig

## 2.3 Load Model & Data
For today's exercise, we'll try to make a smaller version of `SmolLM2` good at English to Danish machine translation

:::{admonition} You can use LoRa for much more than Translation :)
:class: dropdown, tip
As a simple introduction to LoRA, we're doing machine translation, but you can use this approach for anything you'd like really - feel free to switch out the dataset for something you'd like. Or use this notebook as a inspiration for the exam :).

See also this tutorial for instruction-tuning a danish language model using QLoRA -> [Tutorial: Finetuning Language Models](https://www.foundationmodels.dk/blog/2024/02/02/tutorial-finetuning-language-models.html)
:::

We'll load a smaller version of `SmolLM2`:

In [325]:
model_id ="HuggingFaceTB/SmolLM2-135M-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_id)

We'll load the Danish-English translation set, but only a subset with `[:n]` for `n` rows:

In [338]:
n_rows = 2000
train_ds = load_dataset("Helsinki-NLP/opus-100", "da-en", split=f"train[:{n_rows}]")

Let's look at the only column, "translation" to see how it is structured: 

Let's print a few:

In [327]:
for translation in train_ds["translation"][:5]:
    print(f"EN: {translation['en']}")
    print(f"DA: {translation['da']}")
    print()

EN: For the EEA Joint Committee
DA: På Det Blandede EØS-Udvalgs vegne

EN: Metal containing by weight at least 99,9 % of lead, provided that the content by weight of any other element does not exceed the limit specified in the following table:
DA: metal, der indeholder mindst 99,9 vægtprocent bly, forudsat ingen anden bestanddel indgår i mængder, der overstiger de i nedenstående skema anførte grænseværdier:

EN: Think.
DA: Tænk.

EN: Beth...
DA: Beth...

EN: With the Human Hibernation Project, we will be able to save our best men... frozen in their prime, for use when they are needed most.
DA: Vort projekt "Menneskelig dvale" gør det muligt at holde vores bedste mænd nedfrosset i deres bedste tilstand, for at bruge dem efter behov.



## 2.3 Chat Templating
Last week, we played with chat template formatting like this:
```{figure} ../figures/class8/messages_SFT_lora.png
---
name: messages_sft_lora
width: 80%
---
`Messages` dictionary with chat template
```

We want something similar for the task of translating English to Danish. That is, we want a `prompt` containing instructions + an English example & a `desired_response` being the `Danish` example.

### Your Turn: Define a prompt & formatting function
:::{admonition} HANDS-ON
:class: red
1. Create a function called `def format_prompt(example)`
    - It should process a single row `example` in our dataset
    - Define a prompt that contains instructions and an English example. 
    - Format it as a `messages` dictionary, where the `desired_response` is the `Danish` translation.
    - Return the dictionary 

2. Test the function on a single example in `train_ds` & print the result.

3. Use the `map` function (like we did in [class 5](/book/class5/001_finetune.ipynb) when tokenizing) on your `train_ds`, returning `formatted_train_ds`.
:::

#### Solution
Preprocess function (1) + printing one example (2):

In [328]:
def format_prompt(example):
    translation = example["translation"]
    return {"messages": [{"role": "user", "content": f"Translate to Danish: {translation['en']}"}, {"role": "assistant", "content": f"{translation['da']}"}]}

# print one example with train_ds
example = train_ds[0]
formatted_example = format_prompt(example)
print(formatted_example)


{'messages': [{'role': 'user', 'content': 'Translate to Danish: For the EEA Joint Committee'}, {'role': 'assistant', 'content': 'På Det Blandede EØS-Udvalgs vegne'}]}


Use the `map` function:

In [329]:
formatted_train_ds = train_ds.map(format_prompt, batched=False)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

### Final Formatting: Removing the `Translation` Col
We should now have this:

In [330]:
formatted_train_ds

Dataset({
    features: ['translation', 'messages'],
    num_rows: 20000
})

We remove the `translation` column as we don't need it anymore:

In [331]:
formatted_train_ds = formatted_train_ds.remove_columns(["translation"])

## 2.4 LoRa Config & Training
We'll start by configuring LoRA: 

In [332]:
rank = 16
peft_config = LoraConfig(
    r=rank,
    lora_alpha=rank * 2,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

:::{admonition} HANDS-ON
:class: red
Look at the LoRa hyperparameters [here](https://docs.unsloth.ai/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide#hyperparameters-and-recommendations). Try to see if you can make them make sense. Try to google a little bit otherwise :)
:::

Let's define a output_dir:

In [333]:
path = Path.cwd()
output_dir = path.parents[0] / "training" / f".smollm2_da_en_{n_rows}" #n-rows for different train sizes

Then we are ready to train:

In [334]:
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        packing=True, # can speed up training
        chat_template_path=model_id, # use the model's built-in chat template (since we're not tokenizing ourselves)
    ),
    train_dataset=formatted_train_ds,
    peft_config=peft_config
)
trainer.train()

Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-fla

Tokenizing train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

/Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,4.119400
20,4.231900
30,4.111700
40,4.114400
50,4.009300
60,3.975300
70,3.948000
80,3.785000
90,3.764000
100,3.710200


TrainOutput(global_step=730, training_loss=2.755738300166718, metrics={'train_runtime': 1001.9445, 'train_samples_per_second': 1.457, 'train_steps_per_second': 0.729, 'total_flos': 943356955143168.0, 'train_loss': 2.755738300166718, 'epoch': 1.0})

### 2.5 Inference! Testing our Translation LoRa :)
We'll load the final checkpoint:

In [339]:
output_dir = path.parents[0] / "training" / f".smollm2_da_en_{n_rows}"
adapter_path = str(output_dir) + "/checkpoint-60"

# load model with adapter
tokenizer = AutoTokenizer.from_pretrained(adapter_path, local_files_only=True)
model = AutoPeftModelForCausalLM.from_pretrained(adapter_path, device_map="auto", torch_dtype=torch.float16, local_files_only=True)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

Device set to use mps


In [342]:
prompt = "Translate to Danish: I love to drive my car."
format_prompt = pipe.tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
outputs = pipe(format_prompt, max_new_tokens=50, return_full_text=False)
print(outputs[0]["generated_text"])

Danish: "Hall og krøpie gørdelig."


:::{admonition} QUESTION
:class: red
If you don't speak Danish, try to put the answer below into google translate (or ask a Danish speaking friend). Consider if this answer is good.
:::

## 2.5 TV Kitchen: Inference with More Training Examples :)
Perhaps 2000 examples was not enough to train a proper translation machine. For the sake of this class, I didn't want to make you wait 10-30 minutes for training with more examples. 

However, like in a TV-kitchen, I have pre-made LoRa's for you to try in the `resources` folder:
- .smollm2_da_en_2000 / checkpoint-60
- .smollm2_da_en_10000 / checkpoint-357
- .smollm2_da_en_20000 / checkpoint-500
- .smollm2_da_en_50000 / 

:::{admonition} Adhere to my prompt!
:class: important
While you might have defined your own instructions, please note that my prompt is:
```python
prompt = "Translate to Danish: English sentence"
```
In this TV-kitchen, it might affect performance if you deviate from this way of prompting. However, if you are curious, feel free to test if another prompt still works :)
:::

### Your Turn: Test Inference on Various LoRas :)

:::{admonition} HANDS-ON
:class: red
1. Make a function `def test_inference(adapter_path, prompt)` which:
- Takes an adapter_path pointing to one of the pre-made LoRAs 
- Takes a prompt with an instuction and an English sentence to translate 
- Does all of the inference steps above (loading tokenizer, peft model, taking the prompt and applying chat template to it, returns output ...)

2. Test your function on the various LoRa adapters :)
:::

To get you started, you should fix these paths (or optionally, find a smart way to get the last `checkpoint`:)). 

In [ ]:
# fix this with n_rows and correct checkpoint path :)
resources_dir = path.parents[0] / "resources" / "models" / "lora" / f".smollm2_da_en_{n_rows}" #n-rows for different train sizes
adapter_path = str(output_dir) + ".."

### Solution

In [ ]:
def test_inference(adapter_path, prompt):
    # load model with adapter
    print( f"[INFO:] Loading adapter from: {adapter_path}" )
    tokenizer = AutoTokenizer.from_pretrained(adapter_path, local_files_only=True)
    model = AutoPeftModelForCausalLM.from_pretrained(adapter_path, device_map="auto", torch_dtype=torch.float16, local_files_only=True)

    print( f"[INFO:] Running inference for prompt: {prompt}" )
    pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
    format_prompt = pipe.tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    outputs = pipe(format_prompt, max_new_tokens=50, return_full_text=False)
    return outputs[0]["generated_text"]


# try it 
n_rows = 20000
resources_dir = path.parents[0] / "resources" / "models" / "lora" / f".smollm2_da_en_{n_rows}" #n-rows for different train sizes
adapter_path = str(resources_dir) + "checkpoint-730"

## Future work
Things we could have done: 
1. First of all, experimented with the prompt format. I ran all prompts.

Today we have been sort of "vibe-checking" whether the fine-tuning became better, but ideally we should do some sort of benchmarking, such as evaluating a `test set`.

:::{admonition} How do I select parameters for fine-tunign
:class: tip, dropdown
It often requires 
:::
It often requires some experimenting, but we can also become smarter 
For LoRa, specifically, I recommend reading Sebastian Raschka's [blogpost](https://magazine.sebastianraschka.com/p/practical-tips-for-finetuning-llms) (also the one)